In [1]:
from pathlib import Path
import sys

project_root = Path.cwd()
if (project_root / "src").is_dir():
    pass
elif (project_root / "cricket-win-predict" / "src").is_dir():
    project_root = project_root / "cricket-win-predict"
elif project_root.name == "notebooks" and (project_root.parent / "src").is_dir():
    project_root = project_root.parent
else:
    raise FileNotFoundError("Could not locate the cricket-win-predict project root")

sys.path.insert(0, str(project_root))

In [2]:
from src.data.filter import load_data, filter_with_nation_winners
from src.data.preprocess import Dataset

data = filter_with_nation_winners(load_data())
dataset = Dataset(data)

In [3]:
_, second_innings = dataset.get_data_non_sequential(train_test_split=True)

In [4]:
from torch.utils.data import Dataset
import torch

class SecondInningsDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        data_item = self.data[idx]
        input_features = torch.tensor([data_item[k] for k in ["runs", "wickets", "balls"]], dtype=torch.float32)
        labels = torch.tensor(data_item["labels"])
        return { "input": input_features, "labels": labels }

In [29]:
train_data, test_data = [SecondInningsDataset(data) for data in second_innings.values()]
len(train_data), len(test_data)

(418173, 106617)

In [30]:
import torch.nn as nn

class BaselineModel(nn.Module):
    def __init__(self, input_dim, mid_dim=16, dropout=0):
        super(BaselineModel, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, mid_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mid_dim // 2, mid_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mid_dim, mid_dim // 2),
            nn.ReLU(),
            nn.Dropout(),
            nn.Linear(mid_dim // 2, 1),
        )

    def compute_loss(self, pred, labels):
        criterion = nn.BCEWithLogitsLoss()
        return criterion(pred, labels)
    
    def forward(self, x, y):
        pred = self.fc(x)
        loss = self.compute_loss(pred, y)
        return pred, loss
    

In [40]:
from tqdm import tqdm

def train_step(model, batch, optimizer, device):
    model.train()
    optimizer.zero_grad()
    inputs = batch["input"].to(device)
    labels = batch["labels"].float().unsqueeze(1).to(device)
    _, loss = model(inputs, labels)
    loss.backward()
    optimizer.step()
    return loss.item()

def evaluate(model, test_loader, device):
    model.eval()
    total_loss = 0.0
    progress_bar = tqdm(
        test_loader,
        desc="Evaluating",
        unit="batch",
        leave=False,
        dynamic_ncols=True,
    )
    with torch.no_grad():
        for batch in progress_bar:
            inputs = batch["input"].to(device)
            labels = batch["labels"].float().unsqueeze(1).to(device)
            _, loss = model(inputs, labels)
            total_loss += loss.item()
    return total_loss / len(test_loader)

def train_model(model, train_loader, test_loader, optimizer, scheduler, epochs, device):
    model.to(device)
    training_progress = tqdm(
        total=epochs * len(train_loader),
        desc="Training",
        unit="step",
        dynamic_ncols=True,
    )
    train_losses = []
    eval_losses = []
    epoch_summaries = []
    for epoch in range(1, epochs + 1):
        model.train()
        total_train_loss = 0.0

        for batch in train_loader:
            loss = train_step(model, batch, optimizer, device)
            total_train_loss += loss
            training_progress.update(1)
        train_losses.append(total_train_loss / len(train_loader))

        train_loss = total_train_loss / len(train_loader)
        eval_loss = evaluate(model, test_loader, device)
        eval_losses.append(eval_loss)
        epoch_summaries.append(
            f"Epoch {epoch} / {epochs} | "
            f"train_loss={train_loss:.4f} | test_loss={eval_loss:.4f}"
        )
        if scheduler:
            scheduler.step()
    training_progress.close()
    print("\n".join(epoch_summaries))
    return model, train_losses, eval_losses

In [57]:
batch_size = 128
epochs = 10
learning_rate = 0.01
mid_dim = 12
dropout = 0
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Using device: {device}")

Using device: cuda


In [58]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

model = BaselineModel(input_dim=3, mid_dim=mid_dim, dropout=dropout).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.7)

model, train_losses, eval_losses = train_model(model, train_loader, test_loader, optimizer, scheduler, epochs, device)
final_train_loss, final_eval_loss = evaluate(model, train_loader, device), evaluate(model, test_loader, device)
print(f"\n\nFinal train loss: {final_train_loss:.4f}, Final eval loss: {final_eval_loss:.4f}")

Training: 100%|██████████| 32670/32670 [01:32<00:00, 351.55step/s]


Epoch 1 / 10 | train_loss=0.4550 | test_loss=0.3990
Epoch 2 / 10 | train_loss=0.4322 | test_loss=0.3939
Epoch 3 / 10 | train_loss=0.4267 | test_loss=0.3931
Epoch 4 / 10 | train_loss=0.4265 | test_loss=0.3933
Epoch 5 / 10 | train_loss=0.4244 | test_loss=0.3938
Epoch 6 / 10 | train_loss=0.4253 | test_loss=0.3920
Epoch 7 / 10 | train_loss=0.4222 | test_loss=0.3974
Epoch 8 / 10 | train_loss=0.4223 | test_loss=0.3890
Epoch 9 / 10 | train_loss=0.4211 | test_loss=0.3892
Epoch 10 / 10 | train_loss=0.4220 | test_loss=0.3876




Final train loss: 0.3653, Final eval loss: 0.3876
